In [209]:
#the real dataset
from datasets import load_dataset

dataset = load_dataset('fhswf/german_handwriting')


In [210]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['image', 'text'],
        num_rows: 10854
    })
})


In [211]:
print(dataset['train'][0])

{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=3024x547 at 0x14081AEA0>, 'text': '- Terminvorschlag bis'}


In [212]:
subset = dataset['train'].select(range(500))

In [213]:
from PIL import Image

subset = dataset['train'].select(range(500))

widths, heights = [], []
for row in subset:
    w, h = row['image'].size
    widths.append(w)
    heights.append(h)

print("width  min/max/avg:", min(widths), max(widths), sum(widths) / len(widths))
print("height min/max/avg:", min(heights), max(heights), sum(heights) / len(heights))

width  min/max/avg: 569 4080 2457.13
height min/max/avg: 35 547 91.762


In [214]:
vocab = set()
for row in subset:
    vocab.update(row['text'])

print(vocab)
print(len(vocab))

{'"', 'A', 'd', 'ä', '?', 'g', '1', 'F', 'ß', 'z', 'ö', 'm', 'I', 'U', 'M', 'v', 's', '-', '(', ';', '&', '4', 'Z', 'O', 'P', 'E', 'r', 'D', ',', ')', 'b', '+', 'Q', 'f', 'l', 'B', 'R', 'a', 'h', 'e', '2', 'u', '>', '/', 'S', 'V', 'q', 'H', 'N', 'i', ' ', '.', 'Ä', 'L', 'c', 'T', '|', 'k', '7', '9', 'w', 'y', 'J', 'G', 'Ü', 'K', '5', 'n', '0', 'ü', 't', 'C', 'W', '8', '3', 'o', 'p', ':', '=', 'x', 'j', '6', '\\'}
83


In [215]:
#need to get all the images in the dataset for a common height  - checking the median height
import statistics
print(statistics.median(heights))

83.0


In [216]:
#resizig
def resize_to_height(img, target_height = 64):
    orig_w, orig_h = img.size
    scale = target_height / orig_h
    new_w = int(orig_w * scale)
    return img.resize((new_w, target_height))

In [217]:
resized = resize_to_height(subset[0]['image']);print(resized.size)  # resized correctly

(353, 64)


In [218]:
#resizig
def resize_to_height(img, target_height = 64):
    orig_w, orig_h = img.size
    scale = target_height / orig_h
    new_w = int(orig_w * scale)
    return img.resize((new_w, target_height))

In [219]:
#creating dictionaries like lookup tables, 
sorted_vocab = sorted(vocab)
char2idx = {char: idx + 1 for idx, char in enumerate(sorted_vocab)}   # +1 leaves 0 free for blank , this maps characters to numbers
idx2char = {idx: char for char, idx in char2idx.items()} # this maps numbers to characters again

In [220]:
from torch.utils.data import Dataset
import torch 
from torchvision import transforms # ready made image processing steps

to_tensor = transforms.ToTensor() # reusable converter object

class HandwritingDataset(Dataset):
    def __init__(self, hf_dataset, char2idx, target_height=64):
        self.data = hf_dataset
        self.char2idx = char2idx
        self.target_height = target_height

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        image = row['image'].convert('RGB')
        image = resize_to_height(image, self.target_height)
        image_tensor = to_tensor(image)

        label = [self.char2idx[c] for c in row['text']]
        label_tensor = torch.tensor(label, dtype=torch.long)

        return image_tensor, label_tensor




In [221]:
ds = HandwritingDataset(dataset['train'], char2idx)
img_tensor, label_tensor = ds[0] # triggers __getitem__
print(img_tensor.shape, label_tensor)

torch.Size([3, 64, 353]) tensor([ 8,  1, 45, 55, 68, 63, 59, 64, 72, 65, 68, 69, 53, 58, 62, 51, 57,  1,
        52, 59, 69])


In [222]:
batch = [ds[0], ds[1], ds[2]]      # a tiny 3-example "batch", built by hand
images, labels = zip(*batch)        # now images actually exists

widths = [img.shape[2] for img in images]
max_width = max(widths)
print(widths, max_width)

[353, 366, 256] 366


In [223]:
# the image tensors have different widths and the label tensors have different number of
# characters hence not the same dimension

def collate_fn(batch):
    images, labels = zip(*batch)

    ## batch is a list dataloader builds automatically it calls ds[0, 2] lets say 16 times and collects the result into one list of pairs
    ## then should find the widest element in the batch because different sized elements cannot stack on each other

    widths = [img.shape[2] for img in images]  # img.shape on a tensor gives the 3 items that it has and [2] selects the width because its at the end
    max_width = max(widths)

    padded_images = []
    for img in images:
        pad_amount = max_width - img.shape[2]
        padded_img = torch.nn.functional.pad(img, (0, pad_amount))
        padded_images.append(padded_img)

    image_batch = torch.stack(padded_images)

    # doing the same to labels
    label_lengths = [len(label) for label in labels]
    max_label_len = max(label_lengths)

    padded_labels = []
    for label in labels:
        pad_amount = max_label_len - len(label)
        padded_label = torch.nn.functional.pad(label, (0, pad_amount), value=0)
        padded_labels.append(padded_label)

    label_batch = torch.stack(padded_labels)
    label_lengths = torch.tensor(label_lengths, dtype=torch.long)

    return image_batch, label_batch, label_lengths

In [224]:
# actually padding each image
padded_images = []
for img in images : 
    pad_amount = max_width - img.shape[2]
    padded_img = torch.nn.functional.pad(img, (0, pad_amount)) # add 0 pixels to the left side of the width and pad_amout to the right side of the width
    padded_images.append(padded_img) 


print([img.shape for img in padded_images])

[torch.Size([3, 64, 366]), torch.Size([3, 64, 366]), torch.Size([3, 64, 366])]


In [225]:
#turting the list of same- width tensors in to single batch tensor
image_batch = torch.stack(padded_images)
image_batch.shape
#new 3 infront is the batch size 

torch.Size([3, 3, 64, 366])

In [226]:
batch = [ds[0], ds[1], ds[2]]
image_batch, label_batch, label_lengths = collate_fn(batch)

print(image_batch.shape)
print(label_batch.shape)
print(label_lengths)

torch.Size([3, 3, 64, 366])
torch.Size([3, 21])
tensor([21, 19, 11])


In [227]:
#collate_fn works : so now we can wire it into a real dataloader so it runs automatically on shuffled batched isntead of the data bacteches made by hand

from torch.utils.data import DataLoader

loader = DataLoader(ds, batch_size = 8, shuffle=True, collate_fn=collate_fn)
images, labels , label_lengths = next(iter(loader))
print(images.shape)
print(labels.shape)
print(label_lengths)


torch.Size([8, 3, 64, 1714])
torch.Size([8, 59])
tensor([48, 17, 29, 32, 10, 59, 56, 43])


In [228]:
split = dataset['train'].train_test_split(test_size=0.1, seed=42)
train_data = split['train']
val_data = split['test']

print(len(train_data))
print(len(val_data))

9768
1086


In [229]:
#wrapping both into handwritingdataset and build two loaders

train_ds = HandwritingDataset(train_data, char2idx)
val_ds = HandwritingDataset(val_data, char2idx)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, collate_fn=collate_fn)

In [230]:
images, labels, label_lengths = next(iter(train_loader)) # check
print(images.shape, labels.shape)

torch.Size([8, 3, 64, 1972]) torch.Size([8, 84])


Data pipeline is done and i made the data feedable to the model now budiling the model itself

+ CNN — scans the image and picks out useful visual shapes (edges, curves, loops).
+ Slice into columns — chop the image left to right into a sequence of thin strips, like reading it one narrow slice at a time.
+ BiLSTM — reads that sequence of strips in order and guesses, for each strip, "what character is this probably part of."




In [231]:
# CNN for this __init__ and foreward is required
# smallest starting point


import torch.nn as nn 

class CRNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)

    def forward(self, x):
        x = self.conv1(x)
        return x

model = CRNN()
images, labels, label_lengths = next(iter(train_loader))
output = model(images)
print(output.shape)  

torch.Size([8, 16, 64, 2088])


In [247]:
dataset['train'] = dataset['train'].filter(lambda row: row['text'] is not None)
print(len(dataset['train']))

10844


In [248]:
vocab = set()
for row in dataset['train']:
    vocab.update(row['text'])

print(len(vocab))

sorted_vocab = sorted(vocab)
char2idx = {char: idx + 1 for idx, char in enumerate(sorted_vocab)}
idx2char = {idx: char for char, idx in char2idx.items()}

105


In [249]:
#some of the transcriptions are empty so it returns an error so for empty rows i am making a set as bad_rows

bad_rows = []
for i , row in enumerate(dataset['train']):
    if row['text'] is None:
        bad_rows.append(i)

print(len(bad_rows))
print(bad_rows[:10])        

0
[]


In [251]:
train_ds = HandwritingDataset(train_data, char2idx)
val_ds = HandwritingDataset(val_data, char2idx)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, collate_fn=collate_fn)

model = CRNN()
images, labels, label_lengths = next(iter(train_loader))
output = model(images)
print(output.shape)

torch.Size([8, 16, 64, 1633])


In [252]:
#cheking whether the CRNN model runs on this as before

model = CRNN()
images, labels, label_lengths = next(iter(train_loader))
output = model(images)
print(output.shape)

torch.Size([8, 16, 64, 1968])


In [255]:
"""---------------------------------------------------------------------------
RuntimeError                              Traceback (most recent call last)
Cell In[188], line 4
      1 #cheking whether the CRNN model runs on this as before
      2 
      3 model = CRNN()
----> 4 images, labels, label_lengths = next(iter(train_loader))
      5 output = model(images)
      6 print(output.shape)

File /opt/miniconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:725, in _BaseDataLoaderIter.__next__(self)
    722 if self._sampler_iter is None:
    723     # TODO(https://github.com/pytorch/pytorch/issues/76750)
    724     self._reset()  # type: ignore[call-arg]
--> 725 data = self._next_data()
    726 self._num_yielded += 1
    727 if (
    728     self._dataset_kind == _DatasetKind.Iterable
    729     and self._IterableDataset_len_called is not None
    730     and self._num_yielded > self._IterableDataset_len_called
    731 ):

File /opt/miniconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:785, in _SingleProcessDataLoaderIter._next_data(self)
    783 def _next_data(self):
    784     index = self._next_index()  # may raise StopIteration
...
     20 
     21     # doing the same to labels
     22     label_lengths = [len(label) for label in labels]

RuntimeError: stack expects each tensor to be equal size, but got [3, 64, 2104] at entry 0 and [4, 64, 2104] at entry 4
Output is truncated. View as a scrollable element or open in a text editor. Adjust cell output settings... """

#this error says some of the images have 4 insted of 3 RGB ones so this needs to be again sorted out

bad_indices = []
for i in range(len(train_ds)):
    img, label = train_ds[i]
    if img.shape[0] != 3:
        bad_indices.append((i, img.shape))

print(len(bad_indices))
print(bad_indices[:10])

TypeError: 'NoneType' object is not iterable

In [257]:
split = dataset['train'].train_test_split(test_size=0.1, seed=42)
train_data = split['train']
val_data = split['test']

train_ds = HandwritingDataset(train_data, char2idx)
val_ds = HandwritingDataset(val_data, char2idx)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, collate_fn=collate_fn)

print(len(train_data), len(val_data))

9759 1085


In [278]:
model = CRNN()
images, labels, label_lengths = next(iter(train_loader))
output = model(images)
print(output.shape)

torch.Size([8, 16, 64, 2211])
